In [1]:
import pandas as pd
import os
import json
import numpy as np
from os.path import dirname

root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print("CWD:", os.getcwd())
print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

CWD: /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/graphs_repair/


In [2]:
with open("dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [3]:
list(datasets_info.keys())

['BPIC11_f1', 'sepsis_cases_1', 'sepsis_cases_4', 'BPIC15_common']

In [4]:
dataset = "BPIC11_f1" #Select dataset to execute on

In [5]:
tab_all = pd.read_csv(f"datasets/processed/{dataset}_processed_all.csv")
tab_all.head()

,Diagnosis,Treatment code,Diagnosis code,Specialism code,Diagnosis Treatment Combination ID,Age,CaseID,Label,Activity,Producer code,Section,Specialism code.1,group,Number of executions,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases
0,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC410100,SRTH,Section 5,SC61,Radiotherapy,1,1.104692e+09,1380,1,6,23,0.0,0.0,1,5
1,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC419100,SRTH,Section 5,SC61,Radiotherapy,1,1.104692e+09,1380,1,6,23,0.0,0.0,2,5
2,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC10107,SGEH,Section 2,SC7,Nursing ward,1,1.104865e+09,1380,1,1,23,0.0,2880.0,3,5
3,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,339486E,SGEC,Section 2,SC7,Obstetrics & Gynaecology clinic,1,1.104865e+09,1380,1,1,23,0.0,2880.0,4,5
4,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC410100,SGEH,Section 2,SC7,Nursing ward,1,1.104865e+09,1380,1,1,23,0.0,2880.0,5,5


In [6]:
tab_train = pd.read_csv(f"datasets/processed/{dataset}_processed_train.csv")
tab_valid = pd.read_csv(f"datasets/processed/{dataset}_processed_valid.csv")
tab_test = pd.read_csv(f"datasets/processed/{dataset}_processed_test.csv")

In [7]:
#if dataset.startswith("BPIC15"):
#    with open("dataset_features.json", 'r') as file:
#        dataset_info = json.load(file)["BPIC15_common"]
#else:
#    with open("dataset_features.json", 'r') as file:
#        dataset_info = json.load(file)[dataset]


with open("dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]


In [8]:
dataset_info

{'categorical': ['Diagnosis',
  'Treatment code',
  'Diagnosis code',
  'Specialism code',
  'Diagnosis Treatment Combination ID',
  'CaseID',
  'Activity',
  'Producer code',
  'Section',
  'Specialism code.1',
  'group'],
 'numerical': ['Age',
  'Number of executions',
  'timesincemidnight',
  'month',
  'weekday',
  'hour',
  'timesincelastevent',
  'timesincecasestart',
  'event_nr',
  'open_cases']}

In [9]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [10]:
for k in categorical_columns:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")

#cast Label to int (0/1) 
if dataset == "BPI12_DECLINED_COMPLETE":
    for df in (tab_all, tab_train, tab_valid, tab_test):
        df["Label"] = df["Label"].astype(int)  
elif dataset in ["BPIC11_f1"] or dataset.startswith("BPIC15"): #Remove old stuff
    for df in (tab_all, tab_train, tab_valid, tab_test):
        df["Label"] = (df["Label"]=="deviant").astype(int)
else: 
    raise ValueError(f"Unknown dataset name: {dataset!r}")

In [11]:
#Check percentage true/false in test dataset to check it isn't blindly predicting a value.
unique_labels = tab_test.drop_duplicates('CaseID', keep='last')['Label']
percentage_true = unique_labels.mean() * 100

print(f"Test set: {percentage_true:.2f}% True")
print(f"Test set: {100-percentage_true:.2f}% False")

Test set: 47.81% True
Test set: 52.19% False


### Prepare the graphs

In [12]:
import sklearn.preprocessing

from typing import List

In [13]:
def get_case_ids(tab):
    return list(tab["CaseID"].unique())

In [14]:
from torch import tensor, max, int64, float32
from torch_geometric.data import HeteroData

In [15]:
def get_one_hot_encoder(dataset: pd.DataFrame, key: str):
    datas = np.unique(dataset[key].astype(str)).reshape(-1,1)
    onehot = sklearn.preprocessing.OneHotEncoder()
    onehot.fit(datas)
    return onehot

In [16]:
def get_one_hot_encodings(onehot, datas: pd.Series):
    return onehot.transform(datas.reshape(-1, 1)).toarray()

In [17]:
def get_node_features(dataset: pd.DataFrame, trace: pd.DataFrame, cat_features, real_features) -> dict:
 

    res = {}

    for key in trace:
        values = trace[key].values
        if key in cat_features:
            onehot_encoder = get_one_hot_encoder(dataset, key)
            try:
                res[key] = tensor(
                    get_one_hot_encodings(onehot_encoder, values),
                    dtype=float32,
                    requires_grad=True
                )
            except ValueError:
                print(key)
                print(values)
        if key in real_features:
            res[key] = tensor(values,  dtype=float32,requires_grad=True)
            res[key] = res[key].reshape(res[key].shape[0], 1)
        
    

    return res


In [18]:
def compute_edges_indexs(node_features: dict, prefix_len):
    res = {}
    keys = node_features.keys()
    
    indexes = [[i, i + 1] for i in range(prefix_len-1)]
   
    for k in keys:
        if len(node_features[k]) != 1:
            if k == "Activity":
                res[(k, "followed_by", k)] = indexes
                for k2 in keys:
                    if k2 != k:
                        if len(node_features[k2]) == 1:
                            res[(k, "related_to", k2)] = [
                                [i, 0] for i in range(prefix_len)
                            ]
                        else:
                            res[(k, "related_to", k2)] = [
                                [i, i] for i in range(prefix_len)
                            ]
            else:
                res[(k, "related_to", k)] = indexes

    return res

In [19]:

def build_prefixes_graph_from_trace(dataset, trace, cat_features, real_features, prefix_length):
    X = []  # graphs
   
    
    
    node_features = get_node_features(dataset, trace, cat_features, real_features)
    
    
    
    
    G = HeteroData()
        
        
        
    for k in node_features:
        if k != "Label":
            G[k].x = node_features[k][:prefix_length]


    edges_indexes = compute_edges_indexs(node_features, prefix_length)

    


    for k in edges_indexes:
        ce = [[], []]
        for i in range(len(edges_indexes[k])):
            ce[0].append(edges_indexes[k][i][0])
            ce[1].append(edges_indexes[k][i][1])
        edges_indexes[k] = ce

    for k in edges_indexes:
        G[k].edge_index = tensor(edges_indexes[k], dtype=int64)


    ## Get the label of the trace
    label_value = trace["Label"].iloc[0]
    G.y = tensor([int(label_value)], dtype=int64)
             
    X.append(G)
        
    return X

## Create the datasets

In [20]:
case_train_ids = get_case_ids(tab_train)
case_valid_ids = get_case_ids(tab_valid)
case_test_ids = get_case_ids(tab_test)

In [21]:
print(len(case_train_ids))
print(len(case_valid_ids))
print(len(case_test_ids))

729
183
228


In [22]:
tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)

In [23]:
trace = (
        tab_train.query(f"CaseID == '{case_train_ids[0]}'")
        .reset_index()
        .drop(columns="index")
        #.drop(columns="CaseID")
    )
trace 

,Diagnosis,Treatment code,Diagnosis code,Specialism code,Diagnosis Treatment Combination ID,Age,CaseID,Label,Activity,Producer code,Section,Specialism code.1,group,Number of executions,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases
0,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,0,AC410100,SRTH,Section 5,SC61,Radiotherapy,1,1.104692e+09,1380,1,6,23,0.0,0.0,1,5
1,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,0,AC419100,SRTH,Section 5,SC61,Radiotherapy,1,1.104692e+09,1380,1,6,23,0.0,0.0,2,5
2,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,0,AC10107,SGEH,Section 2,SC7,Nursing ward,1,1.104865e+09,1380,1,1,23,0.0,2880.0,3,5
3,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,0,339486E,SGEC,Section 2,SC7,Obstetrics & Gynaecology clinic,1,1.104865e+09,1380,1,1,23,0.0,2880.0,4,5
4,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,0,AC410100,SGEH,Section 2,SC7,Nursing ward,1,1.104865e+09,1380,1,1,23,0.0,2880.0,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,0,AC370000,LBAC,Section 4,SC87,Medical Microbiology,1,1.110395e+09,1380,3,2,23,0.0,95040.0,71,98
71,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,0,370504A,LBAC,Section 4,SC87,Medical Microbiology,1,1.110395e+09,1380,3,2,23,0.0,95040.0,72,98
72,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,0,370505A,LBAC,Section 4,SC87,Medical Microbiology,1,1.110395e+09,1380,3,2,23,0.0,95040.0,73,98
73,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,0,375138A,LBAC,Section 4,SC87,Medical Microbiology,1,1.110395e+09,1380,3,2,23,0.0,95040.0,74,98


In [24]:
import pickle
from tqdm.notebook import tqdm

In [25]:
min_len = tab_all.groupby("CaseID").size().min()
max_len = tab_all.groupby("CaseID").size().max()
print("Minimum trace length:", min_len)
print("Maximum trace length:", max_len)


Minimum trace length: 1
Maximum trace length: 1814


In [26]:
PREFIX_LENGTH = 4

In [27]:
print("Preparing training dataset...")

X_train = []


for i in tqdm(range(len(case_train_ids))):
    trace = (
        tab_train.query(f"CaseID == '{case_train_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH,
        )
        for j in range(len(graphs)):
            X_train.append(graphs[j])

Preparing training dataset...


  0%|          | 0/729 [00:00<?, ?it/s]

In [28]:
print(dataset,"\n")

BPIC11_f1 



In [29]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "wb") as f:
    pickle.dump(X_train, f)

In [30]:
print("Preparing validation dataset...")

X_valid = []


for i in tqdm(range(len(case_valid_ids))):
    trace = (
        tab_valid.query(f"CaseID == '{case_valid_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )
    if len(trace) > PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH
        )
        for i in range(len(graphs)):
            X_valid.append(graphs[i])

Preparing validation dataset...


  0%|          | 0/183 [00:00<?, ?it/s]

In [31]:
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "wb") as f:
    pickle.dump(X_valid, f)

In [32]:
print("Preparing test dataset...")

X_test = []


for i in tqdm(range(len(case_test_ids))):
    trace = (
        tab_test.query(f"CaseID == '{case_test_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH
        )
        for i in range(len(graphs)):
            X_test.append(graphs[i])

Preparing test dataset...


  0%|          | 0/228 [00:00<?, ?it/s]

In [33]:
with open(data_dir_graphs + dataset + "_TEST_repair.pkl", "wb") as f:
    pickle.dump(X_test, f)